# Three-Model Ensemble — XGBoost Training & Hyperparameter Search

Trains three independent XGBoost classifiers, each on a distinct information modality:

| Model | Modality | Features |
|-------|----------|----------|
| **Price** | Technical / market price signals | 8 |
| **Transcript + Report** | LLM-extracted call & filing signals (undecayed ffill + days_since_call) | 10 |
| **News** | Press-release flow & sentiment | 5 |

**Model 2 design note**: NLP features are used in their raw forward-filled form (no call_decay
multiplication). `days_since_call` is included explicitly as the staleness signal, allowing the
tree to learn its own staleness thresholds rather than forcing a fixed exponential decay.
This avoids near-perfect multicollinearity among the 9 evt_* features within each quarter.

**Target**: `target_excess_xfn_5d` — 3-class {−1, 0, +1}, threshold ±0.003  
**Training**: Rolling 2-year window (504 td), quarterly test folds (63 td), 5-day gap  
**Tuning**: Middle folds 4–10 (7 of 13). Criterion: `composite = 0.6 × ASA + 0.4 × Macro F1`  
**Grid**: 108 configs per model — `max_depth` × `min_child_weight` × `n_estimators` × `reg_alpha` × `reg_lambda`  

Artifacts saved to `artifacts/`.

## 1. Imports & Paths

In [1]:
import sys
import warnings
import pickle
import json
from pathlib import Path
from datetime import date
from itertools import product as iproduct

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import f1_score
from IPython.display import display

warnings.filterwarnings('ignore')

NOTEBOOK_DIR = Path('').resolve()
ROOT = NOTEBOOK_DIR.parents[1]
EXP_PARENT = ROOT / 'step3_predictive_model/model_experiments_archive'
for p in [str(ROOT), str(EXP_PARENT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

ARTIFACT_DIR = NOTEBOOK_DIR / 'artifacts'
ARTIFACT_DIR.mkdir(exist_ok=True)
print(f'ROOT          : {ROOT}')
print(f'ARTIFACT_DIR  : {ARTIFACT_DIR}')

ROOT          : /Users/yanyan/Desktop/Projects/AI_driven_company_and_stock_analysis
ARTIFACT_DIR  : /Users/yanyan/Desktop/Projects/AI_driven_company_and_stock_analysis/step3_predictive_model/simple_model_ensemble/artifacts


## 2. Data Load & Feature Engineering

In [2]:
from redesign_single_stock.src.run_redesign_experiments import load_dataset, label_3class
from src.models.walk_forward_config import STRIDE_EVAL

df = load_dataset()

# ── td_return_60d: not in model_features_daily — compute from raw prices ──
td_raw = pd.read_parquet(ROOT / 'data/raw/prices/TD_TO.parquet')
td_raw['date'] = pd.to_datetime(td_raw['date'])
td_raw = td_raw.sort_values('date').reset_index(drop=True)
td_raw['td_return_60d'] = td_raw['adj_close'].pct_change(60)
df = df.merge(td_raw[['date', 'td_return_60d']], on='date', how='left')

# ── Model 1 features (already in df) ─────────────────────────────────────
# (no new engineering needed)

# ── Model 2 — undecayed NLP features (CEO+CFO combined, no call_decay) ───
# Combine CEO + CFO prepared-remarks sentiment into one feature (user requirement).
# All other NLP features used directly from their _ffill columns.
# days_since_call is kept as an explicit staleness feature; the tree learns
# its own staleness thresholds rather than using a fixed exponential decay.
df['exec_tone_ffill'] = df[['transcript_ceo_prep_sentiment_mean_ffill',
                             'transcript_cfo_prep_sentiment_mean_ffill']].mean(axis=1)

# ── Model 3 features (already in df) ─────────────────────────────────────
# (no new engineering needed)

print(f'Loaded {len(df)} rows  |  {df["date"].min().date()} to {df["date"].max().date()}')

Loaded 1285 rows  |  2021-02-25 to 2026-04-09


## 3. Feature Set Definitions

In [3]:
TARGET    = 'target_excess_xfn_5d'
THRESHOLD = 0.003
LABEL_MAP = {-1: 0, 0: 1, 1: 2}
INV_MAP   = {0: -1, 1: 0, 2: 1}
STRIDE    = STRIDE_EVAL   # 5

TRAIN_WINDOW_DAYS = 504   # rolling 2 years of trading days
QUARTER_DAYS      = 63    # 1 quarter of trading days (test window)
GAP_DAYS          = 5     # 5-day gap between train_end and test_start

# ── Model 1 — Price (8 features) ──────────────────────────────────────────
FEATURES_PRICE = [
    'td_return_5d',        # 5-day momentum
    'td_return_20d',       # 20-day trend
    'td_return_60d',       # quarterly momentum
    'td_vs_xfn_5d',        # sector-relative alpha (TD vs XFN)
    'td_dist_52w_high',    # distance from 52-week high (mean-reversion anchor)
    'td_volatility_20d',   # rolling 20d return std
    'td_volume_change_5d', # 5d avg volume vs prior 5d avg
]

# ── Model 2 — Transcript + Report (10 undecayed features) ────────────────
# NLP features in raw forward-filled form. days_since_call provides staleness.
# Features are now genuinely independent within a quarter (no multicollinearity).
FEATURES_TRANSCRIPT = [
    'exec_tone_ffill',                            # combined CEO+CFO prepared-remarks tone
    'transcript_analyst_qa_sentiment_mean_ffill', # analyst Q&A tone
    'framing_gap_ffill',                          # CEO tone vs written filing divergence
    'topic_guidance_share_ffill',                 # guidance discussion share
    'topic_guidance_sentiment_ffill',             # guidance tone
    'topic_regulatory_AML_share_ffill',           # AML/regulatory discussion share
    'topic_regulatory_AML_sentiment_ffill',       # AML/regulatory tone
    'topic_credit_quality_share_ffill',           # credit quality discussion share
    'topic_credit_quality_sentiment_ffill',       # credit quality tone
    'days_since_call',                            # explicit staleness — tree learns own thresholds
]

# ── Model 3 — News (5 features) ───────────────────────────────────────────
FEATURES_NEWS = [
    'news_sent_mean_7d',    # 7-day rolling LLM sentiment
    'news_sent_mean_30d',   # 30-day rolling LLM sentiment
    'news_count_7d',        # 7-day news volume
    'news_count_30d',       # 30-day news volume
    'days_since_last_news', # information recency / silence signal
]

ALL_FEATURES = {
    'price':      FEATURES_PRICE,
    'transcript': FEATURES_TRANSCRIPT,
    'news':       FEATURES_NEWS,
}

# Sanity-check all features exist in df
for model_name, feats in ALL_FEATURES.items():
    missing = [f for f in feats if f not in df.columns]
    assert not missing, f'[{model_name}] Missing columns: {missing}'
    print(f'[{model_name}]  {len(feats)} features — OK')

print(f'\nTarget column present: {TARGET in df.columns}')
print(f'STRIDE = {STRIDE}')

[price]  7 features — OK
[transcript]  10 features — OK
[news]  5 features — OK

Target column present: True
STRIDE = 5


## 4. Quarterly Rolling 2-Year Folds (5-Day Gap)

In [4]:
def make_quarterly_rolling2y_folds(df, target,
                                   train_days=TRAIN_WINDOW_DAYS,
                                   quarter_days=QUARTER_DAYS,
                                   gap=GAP_DAYS):
    """
    Rolling 2-year quarterly folds with a 5-day no-touch gap.

    i           = first test day index
    train_end   = dates[i - 1 - gap]  (prevents leaking the 5d-forward-return target)
    train_start = train_end - train_days  (rolling window)
    """
    dated = df[df[target].notna()].sort_values('date')
    dates = dated['date'].values
    folds = []
    i, fold_id = train_days + gap, 1
    while i < len(dates):
        te_end_i      = min(i + quarter_days - 1, len(dates) - 1)
        train_end_idx = i - 1 - gap
        tr_s_i        = max(0, train_end_idx + 1 - train_days)
        folds.append((
            fold_id,
            str(dates[tr_s_i])[:10],
            str(dates[train_end_idx])[:10],
            str(dates[i])[:10],
            str(dates[te_end_i])[:10],
        ))
        i = te_end_i + 1
        fold_id += 1
    return folds

FOLDS = make_quarterly_rolling2y_folds(df, TARGET)

print(f'Total quarterly folds: {len(FOLDS)}')
print(f'{"Fold":<6} {"Train start":<13} {"Train end":<13} {"Test start":<13} {"Test end":<13}'
      f' {"Gap":>5} {"Train rows":>11} {"Test rows":>9}')
print('-' * 82)
for fold_id, tr_s, tr_e, te_s, te_e in FOLDS:
    tr = df[(df['date'] >= pd.Timestamp(tr_s)) &
            (df['date'] <= pd.Timestamp(tr_e)) & df[TARGET].notna()]
    te = df[(df['date'] >= pd.Timestamp(te_s)) &
            (df['date'] <= pd.Timestamp(te_e)) & df[TARGET].notna()]
    gap_cal = (pd.Timestamp(te_s) - pd.Timestamp(tr_e)).days
    print(f'{fold_id:<6} {tr_s:<13} {tr_e:<13} {te_s:<13} {te_e:<13}'
          f' {gap_cal:>5} {len(tr):>11} {len(te):>9}')

Total quarterly folds: 13
Fold   Train start   Train end     Test start    Test end        Gap  Train rows Test rows
----------------------------------------------------------------------------------
1      2021-02-25    2023-02-28    2023-03-08    2023-06-06        8         504        63


2      2021-05-27    2023-05-30    2023-06-07    2023-09-06        8         504        63
3      2021-08-26    2023-08-29    2023-09-07    2023-12-05        9         504        63
4      2021-11-25    2023-11-28    2023-12-06    2024-03-07        8         504        63
5      2022-02-28    2024-02-29    2024-03-08    2024-06-06        8         504        63
6      2022-05-30    2024-05-30    2024-06-07    2024-09-06        8         504        63
7      2022-08-29    2024-08-29    2024-09-09    2024-12-05       11         504        63
8      2022-11-28    2024-11-28    2024-12-06    2025-03-10        8         504        63
9      2023-03-01    2025-03-03    2025-03-11    2025-06-09        8         504        63
10     2023-05-31    2025-06-02    2025-06-10    2025-09-09        8         504        63
11     2023-08-30    2025-09-02    2025-09-10    2025-12-08        8         504        63
12     2023-11-29    2025-12-01    2025-12-09    2026-03-11        8         504        63

## 5. Training Helpers

In [5]:
def fit_xgb(train_df, test_df, features, params):
    """Fit XGBoost, return (clf, hard_preds, y_true_cls, y_true_cont, prob_matrix)."""
    x_tr   = train_df[features].fillna(train_df[features].median())
    y_tr   = label_3class(train_df[TARGET].values, threshold=THRESHOLD)
    y_enc  = np.array([LABEL_MAP[v] for v in y_tr], dtype=int)

    clf = xgb.XGBClassifier(
        objective='multi:softprob', num_class=3,
        random_state=42, n_jobs=1, verbosity=0, **params
    )
    clf.fit(x_tr, y_enc)

    x_te       = test_df[features].fillna(train_df[features].median())
    y_te_cls   = label_3class(test_df[TARGET].values, threshold=THRESHOLD)
    y_te_cont  = test_df[TARGET].values

    # prob_matrix: shape (n_test, 3) — columns are [P(-1), P(0), P(+1)]
    prob_matrix = clf.predict_proba(x_te)  # rows sum to 1
    hard_preds  = np.array([INV_MAP[int(p)] for p in clf.predict(x_te)])

    return clf, hard_preds, y_te_cls, y_te_cont, prob_matrix


def offset_metrics(preds, y_true_cls, y_true_cont):
    """Stride-offset averaging to de-bias evaluation from fold-boundary effects."""
    rows = []
    for off in range(STRIDE):
        idx = np.arange(off, len(preds), STRIDE)
        p, y_cls, y_cont = preds[idx], y_true_cls[idx], y_true_cont[idx]
        active = (p != 0)
        rows.append({
            'mean_acc':   (p == y_cls).mean(),
            'macro_f1':   f1_score(y_cls, p, average='macro',
                                   zero_division=0, labels=[-1, 0, 1]),
            'active_cov': active.mean(),
            'active_asa': float((np.sign(p[active]) == np.sign(y_cont[active])).mean())
                          if active.sum() > 0 else np.nan,
        })
    return pd.DataFrame(rows).mean()


print('Helpers defined.')

Helpers defined.


## 6. Hyperparameter Grid Search

Grid is designed for ~504 training rows and 5–10 features.  
Shallower trees and higher `min_child_weight` vs the reference 15-feature model to control overfitting.
L1 (`reg_alpha`) and L2 (`reg_lambda`) regularisation are searched explicitly.

**Grid**: 3 × 3 × 3 × 2 × 2 = **108 configs per model**, evaluated on folds 4–10 (7 middle folds).  
**Fixed**: `learning_rate=0.05`, `subsample=0.8`, `colsample_bytree=0.8`, `tree_method='hist'`

In [6]:
GRID = {
    'max_depth':        [2, 3, 4],      # shallower than reference (cap at 4 for small feature sets)
    'min_child_weight': [10, 20, 30],   # conservative: 2–6% of 504 training rows per leaf
    'n_estimators':     [50, 80, 120],  # moderate tree counts
    'reg_alpha':        [0.0, 0.2],     # L1: off or light
    'reg_lambda':       [1.0, 2.0],     # L2: XGBoost default or slightly stronger
}
FIXED_PARAMS = dict(
    learning_rate   = 0.05,
    subsample       = 0.8,
    colsample_bytree= 0.8,
    tree_method     = 'hist',
)

n_folds    = len(FOLDS)
tune_start = max(2, n_folds // 4)          # fold index (0-based), gives fold 4
tune_end   = min(n_folds - 2, n_folds * 3 // 4)  # fold 10
TUNE_FOLDS = FOLDS[tune_start : tune_end + 1]

param_combos = list(iproduct(
    GRID['max_depth'], GRID['min_child_weight'], GRID['n_estimators'],
    GRID['reg_alpha'], GRID['reg_lambda']
))
print(f'Tuning on folds {tune_start+1}..{tune_end+1} ({len(TUNE_FOLDS)} of {n_folds} total)')
print(f'Grid: {len(param_combos)} configs × {len(TUNE_FOLDS)} tuning folds × 3 models')

BEST_PARAMS = {}

for model_name, features in ALL_FEATURES.items():
    print(f'\n── Grid search: {model_name} ({len(features)} features) ──')
    search_rows = []

    for md, mcw, ne, alpha, lam in param_combos:
        params = dict(
            max_depth=md, min_child_weight=mcw, n_estimators=ne,
            reg_alpha=alpha, reg_lambda=lam,
            **FIXED_PARAMS
        )
        fold_metrics = []
        for fold_id, tr_s, tr_e, te_s, te_e in TUNE_FOLDS:
            train_df = df[(df['date'] >= pd.Timestamp(tr_s)) &
                          (df['date'] <= pd.Timestamp(tr_e)) & df[TARGET].notna()]
            test_df  = df[(df['date'] >= pd.Timestamp(te_s)) &
                          (df['date'] <= pd.Timestamp(te_e)) & df[TARGET].notna()]
            if len(test_df) < 10:
                continue
            _, preds, y_cls, y_cont, _ = fit_xgb(train_df, test_df, features, params)
            fold_metrics.append(offset_metrics(preds, y_cls, y_cont))

        if not fold_metrics:
            continue
        agg = pd.DataFrame(fold_metrics).mean()
        search_rows.append({
            'max_depth': md, 'min_child_weight': mcw, 'n_estimators': ne,
            'reg_alpha': alpha, 'reg_lambda': lam,
            'active_asa': agg['active_asa'], 'macro_f1': agg['macro_f1'],
            'active_cov': agg['active_cov'],
            'composite': 0.6 * agg['active_asa'] + 0.4 * agg['macro_f1'],
        })

    search_df = pd.DataFrame(search_rows).sort_values('composite', ascending=False)
    best_row  = search_df.iloc[0]
    best      = dict(
        max_depth        = int(best_row['max_depth']),
        min_child_weight = int(best_row['min_child_weight']),
        n_estimators     = int(best_row['n_estimators']),
        reg_alpha        = float(best_row['reg_alpha']),
        reg_lambda       = float(best_row['reg_lambda']),
        **FIXED_PARAMS
    )
    BEST_PARAMS[model_name] = best

    print('Top-5 configs:')
    display(search_df.head(5).round(4))
    print(f'Best params : {best}')
    print(f'Composite   : {best_row["composite"]:.4f}')

Tuning on folds 4..10 (7 of 13 total)
Grid: 108 configs × 7 tuning folds × 3 models

── Grid search: price (7 features) ──


Top-5 configs:


,max_depth,min_child_weight,n_estimators,reg_alpha,reg_lambda,active_asa,macro_f1,active_cov,composite
67,3,30,80,0.2,2.0,0.5760,0.3057,0.9954,0.4679
66,3,30,80,0.2,1.0,0.5760,0.3038,0.9954,0.4671
65,3,30,80,0.0,2.0,0.5689,0.3016,0.9954,0.4620
102,4,30,80,0.2,1.0,0.5689,0.2991,0.9954,0.4610
27,2,30,50,0.2,2.0,0.5685,0.2992,1.0000,0.4608


Best params : {'max_depth': 3, 'min_child_weight': 30, 'n_estimators': 80, 'reg_alpha': 0.2, 'reg_lambda': 2.0, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'tree_method': 'hist'}
Composite   : 0.4679

── Grid search: transcript (10 features) ──


Top-5 configs:


,max_depth,min_child_weight,n_estimators,reg_alpha,reg_lambda,active_asa,macro_f1,active_cov,composite
90,4,20,80,0.2,1.0,0.5344,0.3167,0.9864,0.4473
56,3,20,120,0.0,1.0,0.5350,0.3088,0.9819,0.4445
57,3,20,120,0.0,2.0,0.5325,0.3037,0.9819,0.4410
91,4,20,80,0.2,2.0,0.5290,0.3087,0.9864,0.4409
88,4,20,80,0.0,1.0,0.5302,0.3062,0.9932,0.4406


Best params : {'max_depth': 4, 'min_child_weight': 20, 'n_estimators': 80, 'reg_alpha': 0.2, 'reg_lambda': 1.0, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'tree_method': 'hist'}
Composite   : 0.4473

── Grid search: news (5 features) ──


Top-5 configs:


,max_depth,min_child_weight,n_estimators,reg_alpha,reg_lambda,active_asa,macro_f1,active_cov,composite
27,2,30,50,0.2,2.0,0.5504,0.2914,1.0000,0.4468
24,2,30,50,0.0,1.0,0.5480,0.2879,1.0000,0.4439
25,2,30,50,0.0,2.0,0.5458,0.2884,1.0000,0.4428
26,2,30,50,0.2,1.0,0.5458,0.2873,1.0000,0.4424
30,2,30,80,0.2,1.0,0.5353,0.2764,0.9978,0.4318


Best params : {'max_depth': 2, 'min_child_weight': 30, 'n_estimators': 50, 'reg_alpha': 0.2, 'reg_lambda': 2.0, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'tree_method': 'hist'}
Composite   : 0.4468


## 7. Save Artifacts

In [7]:
today = date.today().isoformat()

# --- Save best params as JSON ---
params_path = ARTIFACT_DIR / f'best_params_{today}.json'
save_obj = {
    'date': today,
    'target': TARGET,
    'threshold': THRESHOLD,
    'train_window_days': TRAIN_WINDOW_DAYS,
    'quarter_days': QUARTER_DAYS,
    'gap_days': GAP_DAYS,
    'stride': STRIDE,
    'features': ALL_FEATURES,
    'best_params': BEST_PARAMS,
}
with open(params_path, 'w') as f:
    json.dump(save_obj, f, indent=2)
print(f'Best params saved → {params_path}')

# --- Refit each model on the full dataset and save as pickle ---
for model_name, features in ALL_FEATURES.items():
    full_df  = df[df[TARGET].notna()]
    x_full   = full_df[features].fillna(full_df[features].median())
    y_full   = label_3class(full_df[TARGET].values, threshold=THRESHOLD)
    y_enc    = np.array([LABEL_MAP[v] for v in y_full], dtype=int)

    clf_full = xgb.XGBClassifier(
        objective='multi:softprob', num_class=3,
        random_state=42, n_jobs=1, verbosity=0,
        **BEST_PARAMS[model_name]
    )
    clf_full.fit(x_full, y_enc)

    pkl_path = ARTIFACT_DIR / f'xgb_ensemble_{model_name}_{today}.pkl'
    with open(pkl_path, 'wb') as f:
        pickle.dump({'model': clf_full, 'features': features,
                     'params': BEST_PARAMS[model_name]}, f)
    print(f'[{model_name}] Full-data model saved → {pkl_path}')

print('\nAll artifacts saved.')

Best params saved → /Users/yanyan/Desktop/Projects/AI_driven_company_and_stock_analysis/step3_predictive_model/simple_model_ensemble/artifacts/best_params_2026-04-23.json
[price] Full-data model saved → /Users/yanyan/Desktop/Projects/AI_driven_company_and_stock_analysis/step3_predictive_model/simple_model_ensemble/artifacts/xgb_ensemble_price_2026-04-23.pkl
[transcript] Full-data model saved → /Users/yanyan/Desktop/Projects/AI_driven_company_and_stock_analysis/step3_predictive_model/simple_model_ensemble/artifacts/xgb_ensemble_transcript_2026-04-23.pkl
[news] Full-data model saved → /Users/yanyan/Desktop/Projects/AI_driven_company_and_stock_analysis/step3_predictive_model/simple_model_ensemble/artifacts/xgb_ensemble_news_2026-04-23.pkl

All artifacts saved.
